# Proyecto Integrador 2 · Parte V — Detección de comunidades
## Grupos de proveedores que compiten entre sí: Girvan–Newman, espectral, LPA, Louvain y comunidades solapadas

**Qué hace este notebook**
1. Toma el grafo de co-participación del notebook 2 (dos proveedores se conectan si ofertaron en los mismos procesos).
2. Prueba los métodos de partición vistos en clase, evaluados con la **modularidad Q**: **Girvan–Newman**, **clustering espectral**, **LPA** y **Louvain**.
3. Explica **por qué los métodos exactos exigen una submuestra** y cuáles escalan (con tiempos medidos).
4. Corre **Louvain sobre el grafo completo** (40 mil proveedores) y deja las comunidades en un Parquet para el notebook 9.
5. Calcula **comunidades solapadas** con **BigCLAM** sobre un subgrafo.
6. **Interpreta** las comunidades: región, tipo de contratación y qué tan cerradas son.

**Dónde corre cada cosa:** los algoritmos de grafos corren **en local** (NetworkX, NumPy, scikit-learn) porque el grafo cabe en memoria (1.5 millones de aristas). **Spark** se usa solo en el perfil de las comunidades (unir 2.4 millones de participaciones con los procesos).

**Entrada:** `grafo_aristas`, `features_proveedor`, `proveedor_dim` (notebooks 1 y 2) y Parquet del Proyecto 1.
**Salida:** `comunidades_louvain` (Parquet, lo usa el notebook 9) y archivos `com_*` para la página web.

**Pregunta de interés que responde (P3):** ¿qué comunidades forman los proveedores? Cada comunidad es una **agrupación para revisar**, no una acusación: proveedores que coinciden mucho en los mismos procesos pueden ser simplemente del mismo rubro y de la misma región.

**Qué enseñó la clase y qué es extra**
- Girvan–Newman, modularidad Q, clustering espectral, LPA, Louvain: **vistos en clase**.
- **BigCLAM** (comunidades solapadas): el enunciado lo pide y la clase solo lo deja como tema de investigación. Es el **único método extra** de este notebook; se implementa en NumPy, de forma simple, solo sobre el subgrafo.

## 0. Configuración

In [1]:
import os, json, time
import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms import community as nxc

USAR_DRIVE = False   # True si corres en Colab con Drive
RUTA_PARQUET = os.environ.get("P2_PARQUET", "/content/drive/MyDrive/proyecto_integrador_seace/parquet")
RUTA_P2      = os.environ.get("P2_DATOS",   "/content/drive/MyDrive/proyecto_integrador_seace/p2")
WEB          = os.environ.get("P2_WEB",     "/content/drive/MyDrive/proyecto_integrador_seace/web_artefactos/p2")

N_SUB      = 300     # tamaño de la muestra para los métodos exactos (Girvan–Newman, espectral)
MIN_PART_SUB = 50    # la muestra sale de proveedores con al menos 50 participaciones
SEMILLA    = 42
K_MAX_GN   = 60      # Girvan–Newman se detiene al llegar a 60 comunidades
RESOLUCIONES = [0.5, 0.75, 1.0, 1.5, 2.0, 3.0]   # parámetro de resolución de Louvain
RES_BASE   = 1.0     # resolución principal (la de clase)

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(WEB, exist_ok=True)

def guardar_json(nombre, obj):
    with open(f"{WEB}/{nombre}", "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print("guardado:", nombre)

def guardar_csv(nombre, df):
    df.to_csv(f"{WEB}/{nombre}", index=False)
    print("guardado:", nombre, len(df), "filas")

## 1. El grafo
Se lee `grafo_aristas` (aristas con **peso ≥ 2**, es decir, proveedores que coincidieron en al menos 2 procesos). El peso es el número de procesos en común.

In [2]:
aristas = pd.read_parquet(f"{RUTA_P2}/grafo_aristas")
feat = pd.read_parquet(f"{RUTA_P2}/features_proveedor", columns=["proveedor_idx", "ruc", "nombre", "n_participaciones", "categoria_principal"])

G = nx.Graph()
G.add_weighted_edges_from(aristas[["prov_a", "prov_b", "peso"]].itertuples(index=False, name=None))
print(f"grafo completo: {G.number_of_nodes():,} nodos, {G.number_of_edges():,} aristas")

grafo completo: 40,106 nodos, 1,482,754 aristas


## 2. Un subgrafo manejable para los métodos exactos
**Por qué hace falta.** Girvan–Newman recalcula el *edge betweenness* de **todas** las aristas cada vez que quita una, y cada cálculo cuesta del orden de *n·m*. Con 40 mil nodos y 1.5 millones de aristas eso son unos 6·10¹⁰ pasos **por cada arista quitada**. El clustering espectral exige calcular los vectores propios de una matriz de *n × n* (unos 1,600 millones de celdas aquí). Ambos **no caben** en el grafo completo.

**Cómo se arma la muestra (honesta y reproducible):** 300 proveedores elegidos **al azar** (semilla 42) entre los que tienen al menos 50 participaciones, con las aristas que hay entre ellos, y se queda la **componente conexa más grande**. Se elige al azar y no «los más conectados» a propósito: los más conectados forman un bloque casi completo donde ningún método encuentra grupos.

**Limitación:** una muestra aleatoria conserva solo una parte de las aristas; sirve para **comparar métodos**, no para describir el mercado. El mercado se describe con Louvain sobre el grafo completo (sección 7).

In [3]:
rng = np.random.default_rng(SEMILLA)
pool = sorted(n for n in feat[feat.n_participaciones >= MIN_PART_SUB].proveedor_idx if n in G)
muestra = rng.choice(pool, size=N_SUB, replace=False)
S = G.subgraph(muestra).copy()
S = S.subgraph(max(nx.connected_components(S), key=len)).copy()
n_s, m_s = S.number_of_nodes(), S.number_of_edges()
print(f"candidatos: {len(pool):,} | subgrafo: {n_s} nodos, {m_s} aristas, densidad {nx.density(S):.3f}, grado medio {2*m_s/n_s:.1f}")

candidatos: 8,803 | subgrafo: 270 nodos, 828 aristas, densidad 0.023, grado medio 6.1


## 3. Girvan–Newman
**Idea (clase):** repetir *calcular el edge betweenness de cada arista → quitar la mayor → ver si el grafo se partió*. Las aristas con más betweenness son los **puentes** entre grupos. Cada vez que el grafo se parte en más comunidades se mide la **modularidad Q** y se queda el corte con la Q más alta.

**Modularidad Q (clase):** compara la fracción de aristas dentro de las comunidades con la que habría en un **modelo nulo** (grafo con los mismos grados y aristas al azar). Q cerca de 0 = no mejor que el azar; Q más alta = grupos más cerrados que el azar.

Para GN y para comparar los cuatro métodos se usa el grafo **sin pesos** (como en la clase).

In [4]:
t0 = time.time()
curva_gn, mejor_gn = [], (-1, None)
for particion in nxc.girvan_newman(S):
    k = len(particion)
    q = nxc.modularity(S, particion, weight=None)
    curva_gn.append({"k": k, "Q": round(q, 4)})
    if q > mejor_gn[0]:
        mejor_gn = (q, particion)
    if k >= K_MAX_GN:
        break
seg_gn = time.time() - t0
q_gn, part_gn = mejor_gn
print(f"Girvan–Newman: {seg_gn:.0f} s | mejor Q = {q_gn:.3f} con {len(part_gn)} comunidades")
curva_gn = pd.DataFrame(curva_gn)
curva_gn.iloc[::3]

Girvan–Newman: 18 s | mejor Q = 0.578 con 30 comunidades


,k,Q
0,2,0.0537
3,5,0.5208
6,8,0.5545
9,11,0.5678
12,14,0.5673
15,17,0.5682
18,20,0.5653
21,23,0.5640
24,26,0.5627
27,29,0.5607


## 4. ¿Cuánto cuesta crecer? (escalabilidad)
Se mide el tiempo de Girvan–Newman hasta llegar a **10 comunidades** con submuestras de distinto tamaño. Se muestra también cuánto tarda **un solo** cálculo de edge betweenness.

In [5]:
orden = list(S.nodes()); rng2 = np.random.default_rng(SEMILLA); rng2.shuffle(orden)
filas = []
for n in [40, 80, 120, 160, 200, n_s]:
    sub = S.subgraph(orden[:n]).copy()
    t0 = time.time(); nx.edge_betweenness_centrality(sub); uno = time.time() - t0
    t0 = time.time()
    for p in nxc.girvan_newman(sub):
        if len(p) >= 10: break
    filas.append({"nodos": n, "aristas": sub.number_of_edges(), "seg_un_betweenness": round(uno, 3), "seg_gn_hasta_10": round(time.time() - t0, 1)})
escala = pd.DataFrame(filas)
# Estimación para el grafo completo: el costo de un edge betweenness crece como n·m
ult = escala.iloc[-1]
seg_por_unidad = ult.seg_un_betweenness / (ult.nodos * ult.aristas)
est_horas = seg_por_unidad * G.number_of_nodes() * G.number_of_edges() / 3600
print(f"estimación para el grafo completo: un solo edge betweenness tardaría unas {est_horas:.0f} horas (y GN necesita uno por cada arista quitada)")
escala

estimación para el grafo completo: un solo edge betweenness tardaría unas 13 horas (y GN necesita uno por cada arista quitada)


,nodos,aristas,seg_un_betweenness,seg_gn_hasta_10
0,40,12,0.000,0.0
1,80,72,0.002,0.0
2,120,175,0.014,0.0
3,160,265,0.028,0.0
4,200,461,0.072,1.7
5,270,828,0.176,12.3


### Lectura
- El tiempo **crece mucho más rápido que el tamaño**: con 270 nodos GN tarda decenas de veces más que con 200. (Las muestras chicas tienen muy pocas aristas, por eso sus tiempos son casi cero.)
- Con el grafo completo (40,106 nodos y 1.48 millones de aristas) **un solo** edge betweenness tardaría unas 13 horas (estimación a partir de la fórmula *n·m* y el tiempo medido en la muestra), y GN necesita uno por cada arista quitada. Por eso GN **exige una submuestra**.
- Lo que sí escala son los métodos que **no recalculan nada global**: LPA (cada iteración recorre las aristas una vez) y Louvain (mueve nodos entre comunidades y agrupa).

## 5. Clustering espectral
**Idea (clase):** con la **matriz Laplaciana** `L = D − A` (D: grados; A: adyacencia), los **vectores propios** de los valores propios más pequeños separan el grafo en grupos. Se toman los *k* primeros vectores y se agrupan con **K-Means**.

**¿Cuántos grupos?** El **eigengap**: si entre el valor propio *k* y el *k+1* hay un salto grande, *k* es un buen número de grupos.

In [6]:
A = nx.to_numpy_array(S, nodelist=list(S.nodes()), weight=None)
L = np.diag(A.sum(axis=1)) - A
vals, vecs = np.linalg.eigh(L)
idx = list(S.nodes())
from sklearn.cluster import KMeans

def espectral(k):
    lab = KMeans(k, n_init=10, random_state=0).fit_predict(vecs[:, :k])
    return [{idx[i] for i in np.where(lab == c)[0]} for c in range(k)]

filas = []
for k in range(2, 21):
    p = espectral(k)
    filas.append({"k": k, "Q": round(nxc.modularity(S, p, weight=None), 4)})
curva_esp = pd.DataFrame(filas)
k_esp = int(curva_esp.loc[curva_esp.Q.idxmax(), "k"])
q_esp = float(curva_esp.Q.max())
part_esp = espectral(k_esp)

gaps = np.diff(vals[:21])
eigen = pd.DataFrame({"i": range(1, 21), "valor_propio": vals[:20].round(4), "salto_al_siguiente": gaps[:20].round(4)})
k_gap = int(np.argmax(gaps[1:15]) + 2)    # mayor salto después del primer valor propio (que es 0)
print(f"espectral: mejor Q = {q_esp:.3f} con k = {k_esp} | mayor eigengap sugiere k = {k_gap}")
eigen.head(12)

espectral: mejor Q = 0.518 con k = 11 | mayor eigengap sugiere k = 2


,i,valor_propio,salto_al_siguiente
0,1,0.0000,0.1126
1,2,0.1126,0.1228
2,3,0.2354,0.0454
3,4,0.2808,0.0458
4,5,0.3266,0.0449
5,6,0.3715,0.0230
6,7,0.3945,0.0148
7,8,0.4093,0.0468
8,9,0.4561,0.0638
9,10,0.5199,0.0390


## 6. LPA y Louvain
- **LPA** (*Label Propagation*): cada nodo empieza con su propia etiqueta y en cada vuelta **adopta la etiqueta más común entre sus vecinos**. Es muy rápido, pero el resultado **depende de la semilla**.
- **Louvain:** busca **maximizar Q directamente**: mueve nodos a la comunidad vecina que más sube Q y luego trata cada comunidad como un nodo nuevo. No necesita *k*.

Aquí se comparan los cuatro métodos **sobre el mismo subgrafo**, con 10 semillas para LPA y Louvain.

In [7]:
filas = []
for s in range(10):
    t0 = time.time(); p = list(nxc.asyn_lpa_communities(S, weight=None, seed=s)); seg = time.time() - t0
    filas.append({"metodo": "LPA", "semilla": s, "comunidades": len(p), "Q": round(nxc.modularity(S, p, weight=None), 4), "segundos": round(seg, 3)})
for s in range(10):
    t0 = time.time(); p = nxc.louvain_communities(S, weight=None, seed=s); seg = time.time() - t0
    filas.append({"metodo": "Louvain", "semilla": s, "comunidades": len(p), "Q": round(nxc.modularity(S, p, weight=None), 4), "segundos": round(seg, 3)})
semillas = pd.DataFrame(filas)
resumen_sem = semillas.groupby("metodo").agg(Q_min=("Q", "min"), Q_mediana=("Q", "median"), Q_max=("Q", "max"), comunidades_min=("comunidades", "min"), comunidades_max=("comunidades", "max")).round(3)
resumen_sem

,Q_min,Q_mediana,Q_max,comunidades_min,comunidades_max
metodo,,,,,
LPA,0.538,0.566,0.585,20,28
Louvain,0.609,0.613,0.615,7,9


In [8]:
q_lou_sub = semillas[semillas.metodo == "Louvain"].Q.max()
com_lou_sub = len(nxc.louvain_communities(S, weight=None, seed=int(semillas[semillas.metodo == "Louvain"].sort_values("Q").iloc[-1].semilla)))
metodos = pd.DataFrame([
    {"metodo": "Girvan–Newman", "necesita_k": "No (se elige con Q)", "comunidades": len(part_gn), "Q": round(q_gn, 3), "segundos": round(seg_gn, 1), "escala": "No: muy lento"},
    {"metodo": "Espectral", "necesita_k": "Sí (eigengap o Q)", "comunidades": k_esp, "Q": round(q_esp, 3), "segundos": None, "escala": "No: matriz n×n"},
    {"metodo": "LPA (mediana de 10 semillas)", "necesita_k": "No", "comunidades": int(semillas[semillas.metodo == "LPA"].comunidades.median()), "Q": round(semillas[semillas.metodo == "LPA"].Q.median(), 3), "segundos": round(semillas[semillas.metodo == "LPA"].segundos.median(), 3), "escala": "Sí"},
    {"metodo": "Louvain (mejor de 10 semillas)", "necesita_k": "No", "comunidades": com_lou_sub, "Q": round(q_lou_sub, 3), "segundos": round(semillas[semillas.metodo == "Louvain"].segundos.median(), 3), "escala": "Sí"},
])
metodos

,metodo,necesita_k,comunidades,Q,segundos,escala
0,Girvan–Newman,No (se elige con Q),30,0.578,18.000,No: muy lento
1,Espectral,Sí (eigengap o Q),11,0.518,NaN,No: matriz n×n
2,LPA (mediana de 10 semillas),No,23,0.566,0.005,Sí
3,Louvain (mejor de 10 semillas),No,7,0.615,0.008,Sí


### Lectura
- **Louvain** obtiene la **Q más alta** y es el más rápido; además es el que escala.
- **LPA** cambia con la semilla (mire el rango de Q y de comunidades): sirve como **primer paso rápido**, no como resultado final.
- **Girvan–Newman** y **espectral** quedan **por debajo de Louvain** en este subgrafo (Q de 0.58 y 0.52 frente a 0.62) y **exigen** la muestra (GN por tiempo, espectral por la matriz *n × n*). GN llega a su mejor Q con 30 comunidades, muchas de ellas pequeñas; el espectral llega a su mejor Q con 11, aunque el eigengap sugería solo 2.
- Q en este subgrafo es **mucho mayor que 0**: los proveedores **sí forman grupos** más cerrados de lo que daría el azar.

## 7. Louvain sobre el grafo completo
Aquí se usa el grafo **con pesos** (procesos en común). El parámetro **resolución** controla el tamaño de las comunidades: **mayor resolución → comunidades más pequeñas y más numerosas**; menor resolución → pocas y grandes. Con resolución 1 es el Louvain estándar de clase.

También se mide LPA sobre el grafo completo (3 semillas) para mostrar que escala, pero con más variación.

In [9]:
etiquetas = {}      # resolución -> {proveedor_idx: comunidad}, con comunidades numeradas de la más grande (0) a la más chica
filas = []
for r in RESOLUCIONES:
    t0 = time.time()
    comunidades = nxc.louvain_communities(G, weight="weight", resolution=r, seed=SEMILLA)
    seg = time.time() - t0
    comunidades = sorted(comunidades, key=len, reverse=True)
    q = nxc.modularity(G, comunidades, weight="weight", resolution=1.0)
    etiquetas[r] = {n: i for i, c in enumerate(comunidades) for n in c}
    tam = np.array([len(c) for c in comunidades])
    filas.append({"resolucion": r, "comunidades": len(comunidades), "Q": round(q, 4), "mayor": int(tam[0]),
                  "pct_en_las_10_mayores": round(100 * tam[:10].sum() / tam.sum(), 1),
                  "comunidades_de_10_o_mas": int((tam >= 10).sum()),
                  "comunidades_de_menos_de_10": int((tam < 10).sum()), "segundos": round(seg, 1)})
    print(f"resolución {r}: {len(comunidades)} comunidades, Q = {q:.3f}, {seg:.0f} s")
por_res = pd.DataFrame(filas)
por_res

resolución 0.5: 131 comunidades, Q = 0.599, 23 s


resolución 0.75: 130 comunidades, Q = 0.640, 25 s


resolución 1.0: 135 comunidades, Q = 0.648, 27 s


resolución 1.5: 142 comunidades, Q = 0.643, 28 s


resolución 2.0: 145 comunidades, Q = 0.633, 27 s


resolución 3.0: 164 comunidades, Q = 0.597, 28 s


,resolucion,comunidades,Q,mayor,pct_en_las_10_mayores,comunidades_de_10_o_mas,comunidades_de_menos_de_10,segundos
0,0.50,131,0.5994,14735,98.7,16,115,22.5
1,0.75,130,0.6402,11348,95.0,18,112,24.5
2,1.00,135,0.6483,7219,83.2,22,113,27.4
3,1.50,142,0.6434,6960,75.0,30,112,28.2
4,2.00,145,0.6330,6439,65.6,34,111,26.5
5,3.00,164,0.5972,3496,49.5,53,111,27.7


In [10]:
filas = []
for s in range(3):
    t0 = time.time(); p = list(nxc.asyn_lpa_communities(G, weight="weight", seed=s)); seg = time.time() - t0
    filas.append({"semilla": s, "comunidades": len(p), "Q": round(nxc.modularity(G, p, weight="weight"), 4), "mayor": max(len(c) for c in p), "segundos": round(seg, 1)})
lpa_completo = pd.DataFrame(filas)
lpa_completo

,semilla,comunidades,Q,mayor,segundos
0,0,462,0.5866,17694,90.5
1,1,480,0.5934,15465,74.5
2,2,463,0.5792,17963,87.5


### Lectura
- **Louvain procesa 1.5 millones de aristas en unos 25 segundos** en esta máquina; Girvan–Newman no podría. Esa es la respuesta práctica a la escalabilidad.
- La Q del grafo completo es **alta** (0.65 con resolución 1): hay estructura real de grupos.
- Casi todas las «comunidades» de la tabla son **islas diminutas** (menos de 10 proveedores): el grafo tiene unas 100 componentes pequeñas separadas del bloque principal. Por eso se cuenta aparte cuántas comunidades tienen al menos 10 proveedores: son unas 16 con resolución 0.5 y unas 53 con resolución 3.
- **A mayor resolución hay más comunidades relevantes y más pequeñas** (la mayor baja de 14,735 a 3,496 proveedores); la Q medida con la definición estándar es máxima cerca de resolución 1, que es lo que optimiza Louvain, y baja a los lados. No hay una resolución «correcta»: depende de qué tan fino se quiera ver el mercado.
- **LPA** sobre el grafo completo da una Q menor y deja una comunidad gigante: es útil para una primera mirada, pero Louvain es mejor aquí.

In [11]:
# Se guardan las comunidades con la resolución base: las usa el notebook 9 (embeddings)
lab = pd.DataFrame({"proveedor_idx": list(etiquetas[RES_BASE].keys()), "comunidad": list(etiquetas[RES_BASE].values())})
os.makedirs(f"{RUTA_P2}/comunidades_louvain", exist_ok=True)
lab.to_parquet(f"{RUTA_P2}/comunidades_louvain/part-0.parquet", index=False)
print("comunidades_louvain guardado:", len(lab), "proveedores,", lab.comunidad.nunique(), "comunidades")

comunidades_louvain guardado: 40106 proveedores, 135 comunidades


## 8. Interpretar las comunidades
Para describir cada comunidad se calcula, **por proveedor**, dónde y en qué rubro ofertó:
- **Departamento dominante:** el departamento de las entidades de los procesos en que ofertó (el más frecuente).
- **Categoría dominante:** bienes (`goods`), servicios (`services`) u obras (`works`) más frecuente.

Y **por comunidad**:
- **Cohesión:** de todas las conexiones de sus miembros, qué parte queda **dentro** de la comunidad (1 = comunidad cerrada; cerca de 0 = sus miembros se conectan sobre todo con otros grupos).
- **% en el departamento principal** y **% en la categoría principal:** qué tan «regional» o «de un solo rubro» es.

Estos cálculos usan la unión de 2.4 millones de participaciones con los procesos, por eso se hacen en **Spark**.

In [12]:
# Solo Colab: !pip install -q pyspark==3.5.3
from pyspark.sql import SparkSession, functions as F, Window
spark = (SparkSession.builder.master("local[*]").appName("p2_comunidades")
         .config("spark.driver.memory", "4g").config("spark.sql.shuffle.partitions", "16").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

dim  = spark.read.parquet(f"{RUTA_P2}/proveedor_dim").select("proveedor_idx", "ruc")
post = spark.read.parquet(f"{RUTA_PARQUET}/postores").select("release_id", "ruc")
proc = spark.read.parquet(f"{RUTA_PARQUET}/procesos").select("release_id", "categoria", "entidad_departamento")
part = (post.join(dim, "ruc").select("release_id", "proveedor_idx").distinct().join(proc, "release_id"))

def moda(col, nombre):
    c = part.groupBy("proveedor_idx", col).agg(F.count("*").alias("n"))
    w = Window.partitionBy("proveedor_idx").orderBy(F.desc("n"), col)
    return c.withColumn("pos", F.row_number().over(w)).filter("pos = 1").select("proveedor_idx", F.col(col).alias(nombre))

atrib = moda("entidad_departamento", "departamento").join(moda("categoria", "categoria"), "proveedor_idx").toPandas()
print("proveedores con perfil:", len(atrib))
os.makedirs(f"{RUTA_P2}/perfil_proveedor", exist_ok=True)
atrib.to_parquet(f"{RUTA_P2}/perfil_proveedor/part-0.parquet", index=False)    # lo usa el notebook 9
spark.stop()

To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


proveedores con perfil: 104617


In [13]:
atrib = atrib.set_index("proveedor_idx")
grado_pond = pd.concat([aristas.groupby("prov_a").peso.sum(), aristas.groupby("prov_b").peso.sum()]).groupby(level=0).sum()
nombres = feat.set_index("proveedor_idx")

def perfil(etq, top=15):
    s = pd.Series(etq, name="comunidad")
    comp = aristas.assign(ca=aristas.prov_a.map(s), cb=aristas.prov_b.map(s))
    interno = comp[comp.ca == comp.cb].groupby("ca").peso.sum()
    fuerza = grado_pond.groupby(s).sum()
    filas = []
    for c, tam in s.value_counts().head(top).items():
        miembros = s[s == c].index
        dep = atrib.reindex(miembros).departamento.value_counts(normalize=True)
        cat = atrib.reindex(miembros).categoria.value_counts(normalize=True)
        lider = grado_pond.reindex(miembros).sort_values(ascending=False).head(3).index
        filas.append({
            "comunidad": int(c), "proveedores": int(tam),
            "cohesion": round(float(2 * interno.get(c, 0) / fuerza[c]), 3),
            "departamento_principal": dep.index[0].title() if len(dep) else None, "pct_departamento": round(100 * float(dep.iloc[0]), 1) if len(dep) else None,
            "categoria_principal": cat.index[0] if len(cat) else None, "pct_categoria": round(100 * float(cat.iloc[0]), 1) if len(cat) else None,
            "participaciones_medianas": float(nombres.reindex(miembros).n_participaciones.median()),
            "lideres_idx": list(lider),
        })
    return pd.DataFrame(filas)

pf_base = perfil(etiquetas[RES_BASE])
pf_base.drop(columns="lideres_idx")

,comunidad,proveedores,cohesion,departamento_principal,pct_departamento,categoria_principal,pct_categoria,participaciones_medianas
0,0,7219,0.813,Lima,21.5,works,80.5,19.0
1,1,6158,0.667,Lima,33.0,goods,84.1,15.0
2,2,4335,0.821,Lima,26.1,services,96.5,18.0
3,3,3549,0.561,Lima,34.9,services,82.6,19.0
4,4,3095,0.680,Cusco,25.4,services,70.8,18.0
5,5,2002,0.766,Lima,61.1,services,96.4,15.0
6,6,1867,0.760,Lima,69.6,services,52.2,15.0
7,7,1780,0.911,Lima,30.6,goods,98.3,18.0
8,8,1678,0.873,Lima,71.4,goods,84.3,24.0
9,9,1670,0.735,Cusco,29.9,goods,94.3,13.0


### Lectura
- Las comunidades grandes se separan sobre todo por **rubro**: en las 15 mayores, la categoría principal (obras, bienes o servicios) reúne entre 52% y 99% de los miembros, y en 12 de ellas supera el 80%.
- **La región pesa menos:** Lima es el departamento principal en 11 de las 15 (porque concentra muchos procesos), con entre 14% y 75% de los miembros según la comunidad; en las otras 4 (Cusco y Ayacucho) el departamento principal reúne solo entre 14% y 35%, es decir, mezclan proveedores de varios departamentos.
- Una **cohesión alta** significa que sus miembros coinciden **sobre todo entre sí**; una cohesión baja, que la comunidad es más bien una zona de paso entre grupos.
- Una comunidad concentrada en **un solo departamento y una sola categoría** es lo esperable de un **mercado local** (por ejemplo, obras en una región): proveedores del mismo rubro y lugar se encuentran en los mismos procesos **sin que haya nada raro**.
- Una comunidad es una **agrupación a revisar con más contexto** (quién compra, qué compra, en qué fechas), **nunca una prueba de acuerdos entre proveedores**.

## 9. Comunidades solapadas: BigCLAM (AGM)
**Idea.** En una partición, cada proveedor está en **una** comunidad. En la realidad un proveedor puede participar en **varios mercados** a la vez (por ejemplo, vender bienes en dos regiones). El modelo **AGM** (grafo de afiliación a comunidades) dice: dos nodos se conectan con más probabilidad cuantas más comunidades comparten. **BigCLAM** lo hace continuo: cada nodo tiene un vector **F** (una «fuerza de pertenencia» por comunidad, ≥ 0) y

`P(u y v conectados) = 1 − exp(−Fᵤ · Fᵥ)`.

Se busca F que **maximice la verosimilitud** (que la red observada sea lo más probable posible) con **ascenso por gradiente**. Un nodo pertenece a la comunidad *c* si `F[u, c]` supera un umbral δ.

**Implementación simple** (NumPy, sobre el subgrafo de 270 nodos; la matriz es pequeña). Es el único método **extra** de este notebook.

In [14]:
Asub = A.copy()                       # matriz de adyacencia del subgrafo (sin pesos), mismo orden que idx
n = Asub.shape[0]
NE = 1 - Asub - np.eye(n)             # pares sin arista (sin la diagonal)

def loglik(F):
    X = F @ F.T
    return 0.5 * (np.sum(Asub * np.log(1 - np.exp(-X) + 1e-12)) - np.sum(NE * X))

def bigclam(C, iteraciones=1500, semilla=0):
    r = np.random.default_rng(semilla)
    F = r.uniform(0.0, 0.1, (n, C))
    ll, paso = loglik(F), 0.01
    for _ in range(iteraciones):
        X = F @ F.T
        W = Asub * np.exp(-X) / (1 - np.exp(-X) + 1e-12)
        grad = W @ F - NE @ F
        # búsqueda de línea: se reduce el paso hasta que la verosimilitud suba
        for _ in range(25):
            F_nuevo = np.maximum(F + paso * grad, 1e-8)
            ll_nuevo = loglik(F_nuevo)
            if ll_nuevo > ll:
                F, ll, paso = F_nuevo, ll_nuevo, paso * 1.3
                break
            paso *= 0.5
        else:
            break
    return F, ll

eps = 2 * m_s / (n * (n - 1))
delta = np.sqrt(-np.log(1 - eps))     # umbral de pertenencia (BigCLAM)
print(f"umbral de pertenencia δ = {delta:.3f}")

def mejor_de_3(C):
    """El ascenso por gradiente puede quedar en un óptimo local: se prueban 3 semillas y se queda la de mayor verosimilitud."""
    corridas = [bigclam(C, semilla=s) for s in range(3)]
    return max(corridas, key=lambda x: x[1])

filas = []
for C in [4, 8, 12, 16]:
    F, ll = mejor_de_3(C)
    M_ = F > delta
    n_com = int((M_.sum(axis=0) >= 3).sum())
    n_multi = int((M_.sum(axis=1) >= 2).sum())
    sin = int((M_.sum(axis=1) == 0).sum())
    filas.append({"C": C, "log_verosimilitud": round(ll, 1), "comunidades_con_3_o_mas": n_com,
                  "nodos_en_2_o_mas": n_multi, "pct_solapados": round(100 * n_multi / n, 1), "nodos_sin_comunidad": sin})
bc_C = pd.DataFrame(filas)
bc_C

umbral de pertenencia δ = 0.152


,C,log_verosimilitud,comunidades_con_3_o_mas,nodos_en_2_o_mas,pct_solapados,nodos_sin_comunidad
0,4,-2260.5,4,10,3.7,130
1,8,-1762.1,8,22,8.1,90
2,12,-1504.2,12,28,10.4,84
3,16,-1340.0,16,45,16.7,59


In [15]:
# C = 8: cercano al número de comunidades de Louvain en este subgrafo (para poder comparar).
# La verosimilitud siempre mejora al subir C, así que no sirve para elegirlo; con C grande las comunidades se fragmentan.
C_BC = 8
F, ll = mejor_de_3(C_BC)
M_ = F > delta
grupos = [set(np.array(idx)[M_[:, c]]) for c in range(C_BC) if M_[:, c].sum() >= 3]
membresias = M_.sum(axis=1)
print(f"C = {C_BC}: {len(grupos)} comunidades con ≥ 3 nodos | {int((membresias >= 2).sum())} nodos ({100*(membresias >= 2).mean():.1f}%) en 2 o más | {int((membresias == 0).sum())} sin comunidad")

# Comparación con Louvain (misma red): cada nodo se asigna a su comunidad de mayor F
from sklearn.metrics import adjusted_rand_score
lou_sub = nxc.louvain_communities(S, weight=None, seed=int(semillas[semillas.metodo == "Louvain"].sort_values("Q").iloc[-1].semilla))
lou_lab = {nn: i for i, c in enumerate(lou_sub) for nn in c}
dom = F.argmax(axis=1)
ari_bc = adjusted_rand_score([lou_lab[nn] for nn in idx], dom)
print(f"ARI entre la comunidad dominante de BigCLAM y la de Louvain: {ari_bc:.3f}")

# Observación de clase: ¿la densidad es mayor en las zonas de solapamiento?
def densidad(nodos_):
    nodos_ = list(nodos_)
    return nx.density(S.subgraph(nodos_)) if len(nodos_) > 1 else float("nan")
solapados = {nn for nn, k in zip(idx, membresias) if k >= 2}
dens_sol = densidad(solapados)
dens_com = float(np.mean([densidad(g) for g in grupos]))
print(f"densidad entre nodos solapados: {dens_sol:.3f} | densidad media dentro de comunidades: {dens_com:.3f} | densidad de todo el subgrafo: {nx.density(S):.3f}")

C = 8: 8 comunidades con ≥ 3 nodos | 22 nodos (8.1%) en 2 o más | 90 sin comunidad
ARI entre la comunidad dominante de BigCLAM y la de Louvain: 0.524
densidad entre nodos solapados: 0.156 | densidad media dentro de comunidades: 0.235 | densidad de todo el subgrafo: 0.023


In [16]:
meta = feat.set_index("proveedor_idx")
sol = pd.DataFrame({"proveedor_idx": idx, "comunidades": membresias, "grado": [S.degree(nn) for nn in idx]})
sol = sol[sol.comunidades >= 2].sort_values(["comunidades", "grado"], ascending=False).head(15)
sol["nombre"] = sol.proveedor_idx.map(meta.nombre); sol["ruc"] = sol.proveedor_idx.map(meta.ruc)
sol["departamento"] = sol.proveedor_idx.map(atrib.departamento).str.title(); sol["categoria"] = sol.proveedor_idx.map(atrib.categoria)
sol[["nombre", "ruc", "comunidades", "grado", "departamento", "categoria"]]

,nombre,ruc,comunidades,grado,departamento,categoria
8,AUTORES DEL PERU S.A.C.,20600731531,3,17,Puno,goods
20,MAWI SUPPLY SOCIEDAD ANONIMA CERRADA,20544298942,3,11,Lima,services
94,JBG CONSTRUCTORES SOCIEDAD ANONIMA CERRADA - J...,20555976268,2,43,Lima,services
203,C & R EJECUTORES S.A.C.,20557038338,2,31,Lima,works
75,L Y H CONSULTORES Y EJECUTORES E.I.R.L.,20602239986,2,22,Lambayeque,services
207,IMPERIO K Y M S.R.L.,20601608341,2,19,Cusco,goods
225,DRELCA CONTRATISTAS GENERALES SOCIEDAD COMERCI...,20602034390,2,18,Puno,services
194,CONSTRUCTORA JHAD S.A.C,20482530592,2,15,Lambayeque,services
214,CONIESA E.I.R.L.,20322446374,2,15,Lima,works
53,CONSATEL S.A.C.,20547508972,2,13,Lima,goods


### Lectura
- **Una parte de los proveedores pertenece a más de una comunidad:** los que ofertan en varios rubros o regiones **unen** comunidades. La partición de Louvain obliga a elegir una sola.
- **Muchos proveedores quedan «sin comunidad»:** son los de pocas conexiones en la muestra (periferia); su fuerza de pertenencia no llega al umbral. No significa que no compitan, solo que la muestra no tiene datos suficientes de ellos.
- **La verosimilitud mejora al subir C** (mire la tabla) porque el modelo tiene más parámetros; por eso C no se elige por ese número, sino por interpretabilidad.
- **BigCLAM y Louvain coinciden en parte** (ARI moderado): ven la misma estructura general, pero BigCLAM permite que un mismo proveedor esté en varios grupos.
- **La observación de clase no se cumple aquí:** la densidad entre los proveedores solapados (0.16) es **menor** que dentro de las comunidades (0.24), aunque mucho mayor que la del subgrafo completo (0.023). Con solo 22 nodos solapados y una red tan dispersa, el resultado es poco concluyente.
- Es una implementación simple sobre **270 nodos**; escalar BigCLAM a todo el grafo exigiría la versión por bloques del paper original, que no se hace aquí.

## 10. Archivos para la página web
**Datos personales:** el RUC que no empieza con 20 se enmascara (`10283****79`) y el nombre de una persona natural (RUC 10, 15, 16 o 17) se reemplaza por «Persona natural».

In [17]:
def enmascarar(ruc):
    r = str(ruc); return r if r.startswith("20") else r[:5] + "****" + r[-2:]
def nombre_visible(ruc, nombre):
    r = str(ruc); return f"Persona natural (RUC {enmascarar(r)})" if r[:2] in ("10", "15", "16", "17") else nombre
def publico(df):
    d = df.copy()
    d["nombre"] = [nombre_visible(r, nn) for r, nn in zip(d.ruc, d.nombre)]
    d["ruc"] = d.ruc.map(enmascarar)
    return d

guardar_csv("com_gn_curva.csv", curva_gn)
guardar_csv("com_espectral.csv", curva_esp)
guardar_csv("com_eigenvalores.csv", eigen)
guardar_csv("com_escala.csv", escala)
guardar_csv("com_semillas.csv", semillas)
guardar_csv("com_metodos.csv", metodos)
guardar_csv("com_resolucion.csv", por_res)
guardar_csv("com_lpa_completo.csv", lpa_completo)
guardar_csv("com_bigclam_C.csv", bc_C)

# perfil por resolución (las 10 comunidades más grandes de cada una) con los 3 proveedores más conectados
filas = []
for r in RESOLUCIONES:
    p = perfil(etiquetas[r], top=10)
    for _, fila in p.iterrows():
        lideres = [nombre_visible(meta.loc[i, "ruc"], meta.loc[i, "nombre"]) for i in fila.lideres_idx]
        filas.append({**{k: v for k, v in fila.items() if k != "lideres_idx"}, "resolucion": r, "lideres": " · ".join(lideres)})
guardar_csv("com_perfil.csv", pd.DataFrame(filas))

# tamaños de todas las comunidades con la resolución base (para el gráfico de cola larga)
tam = pd.Series(etiquetas[RES_BASE]).value_counts().reset_index(); tam.columns = ["comunidad", "proveedores"]
guardar_csv("com_tamanos.csv", tam)

sol_web = publico(sol[["nombre", "ruc", "comunidades", "grado", "departamento", "categoria"]])
guardar_csv("com_solapados.csv", sol_web)

guardar_json("p5_resumen.json", {
    "grafo": {"nodos": G.number_of_nodes(), "aristas": G.number_of_edges()},
    "subgrafo": {"descripcion": f"{N_SUB} proveedores al azar (semilla {SEMILLA}) entre los de ≥ {MIN_PART_SUB} participaciones; componente conexa mayor.",
                 "nodos": n_s, "aristas": m_s, "densidad": round(nx.density(S), 4)},
    "gn": {"mejor_Q": round(q_gn, 4), "comunidades": len(part_gn), "segundos": round(seg_gn, 1)},
    "escala": {"horas_estimadas_un_betweenness_grafo_completo": round(float(est_horas), 1)},
    "espectral": {"mejor_Q": round(q_esp, 4), "k": k_esp, "k_eigengap": k_gap},
    "louvain_completo_resolucion_1": por_res[por_res.resolucion == RES_BASE].iloc[0].to_dict(),
    "bigclam": {"C": C_BC, "umbral_delta": round(float(delta), 4), "comunidades_con_3_o_mas": len(grupos),
                "nodos_solapados": int((membresias >= 2).sum()), "pct_solapados": round(100 * float((membresias >= 2).mean()), 1),
                "ari_vs_louvain": round(float(ari_bc), 3), "densidad_solapados": round(dens_sol, 3),
                "densidad_media_comunidades": round(dens_com, 3), "densidad_subgrafo": round(nx.density(S), 3)},
})

guardado: com_gn_curva.csv 59 filas
guardado: com_espectral.csv 19 filas
guardado: com_eigenvalores.csv 20 filas
guardado: com_escala.csv 6 filas
guardado: com_semillas.csv 20 filas
guardado: com_metodos.csv 4 filas
guardado: com_resolucion.csv 6 filas
guardado: com_lpa_completo.csv 3 filas
guardado: com_bigclam_C.csv 4 filas


guardado: com_perfil.csv 60 filas
guardado: com_tamanos.csv 135 filas
guardado: com_solapados.csv 15 filas
guardado: p5_resumen.json
